# Module 6：DWA 宽表即席查询

## 痛点故事

> **「等 IT 排期 3 天才能拿到数据」的尬**
>
> 月底分析会上，业务问「10 月各矿井日产量是多少？」
> 数据分析师翻出 VBAK、PI tags、LIMS samples 三套原始数据，
> 要写 5 个 JOIN、等 IT 排期 3~5 天。
>
> **DWA 解法**：有了 DWA 预聚合宽表，10 秒钟出结果，不用再等 IT 排期。

---

## 技术选型：为什么用 DuckDB

| 维度 | DuckDB（本次） | ClickHouse |
|------|---------------|-----------|
| 部署模式 | 嵌入式，无需服务进程 | 分布式服务器 |
| 适用规模 | 单节点 <100GB（本项目 ~1GB） | TB~PB 级集群 |
| 性能 | ClickBench 2025年10月排名第一 | 多节点并行 |
| 运维成本 | 零运维（pip install duckdb） | 需管理集群 |

> **Phase 2 升级路径**：数据规模增长到 100GB+ 时，升级为 ClickHouse / Doris，SQL 语法兼容。


In [ ]:
import subprocess, sys, os
try:
    import duckdb
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'duckdb'], check=True)
    import duckdb

ROOT = "/home/szs/Playground/dg-demo"
LAKEHOUSE = f"{ROOT}/data/lakehouse"
conn = duckdb.connect()
print(f'LAKEHOUSE = {LAKEHOUSE}')


In [ ]:
# 检查 DWA 宽表是否已生成
missing = [p for p in [
    f"{LAKEHOUSE}/dwa/sap_erp/dwa_sales_daily",
    f"{LAKEHOUSE}/dwa/pi_system/dwa_tag_alarm",
    f"{LAKEHOUSE}/dwa/lims/dwa_coal_quality",
] if not os.path.exists(p)]
if missing:
    print("❌ DWA 表缺失，请先运行：")
    print("   uv run python scripts/build_dwa_models.py --layer dwa")
    raise SystemExit(1)
print("✅ DWA 宽表检查通过（3/3 存在）")


## 步骤 2：即席查询 — 日销售趋势

In [ ]:
conn.execute(f"CREATE VIEW IF NOT EXISTS dwa_sales AS "
    f"SELECT * FROM delta_scan('{LAKEHOUSE}/dwa/sap_erp/dwa_sales_daily')")
df = conn.execute('''
    SELECT sale_date, order_count, customer_count,
           ROUND(total_amount/1e6, 2) AS total_amount_m
    FROM dwa_sales ORDER BY sale_date ASC LIMIT 10
''').df()
print('最近 10 天日销售汇总（百万元）：')
print(df.to_string(index=False))


## 步骤 3：即席查询 — 传感器告警 Top20

In [ ]:
conn.execute(f"CREATE VIEW IF NOT EXISTS dwa_alarm AS "
    f"SELECT * FROM delta_scan('{LAKEHOUSE}/dwa/pi_system/dwa_tag_alarm')")
df = conn.execute('''
    SELECT mine, tag, high_value_count, missing_count
    FROM dwa_alarm ORDER BY high_value_count DESC LIMIT 10
''').df()
print('Top10 高频告警传感器：')
print(df.to_string(index=False))


## 步骤 4：即席查询 — 月度煤质报告

In [ ]:
conn.execute(f"CREATE VIEW IF NOT EXISTS dwa_quality AS "
    f"SELECT * FROM delta_scan('{LAKEHOUSE}/dwa/lims/dwa_coal_quality')")
df = conn.execute('''
    SELECT MINE_CODE, month, SAMPLE_TYPE, sample_count,
           avg_ash_content, avg_gross_calorific
    FROM dwa_quality ORDER BY month DESC, MINE_CODE LIMIT 10
''').df()
print('月度煤质报告（最近月份）：')
print(df.to_string(index=False))


## 步骤 5：4 个分析场景验证

验证 DWA 宽表对 4 个典型业务场景的支撑能力。

### 场景 1：销售趋势

In [ ]:
df1 = conn.execute('''
    SELECT sale_date, order_count,
           ROUND(total_amount/1e6, 2) AS amt_m
    FROM dwa_sales ORDER BY sale_date DESC LIMIT 10
''').df()
print(f'场景1 - 销售趋势：{len(df1)} 条记录')
print(df1.to_string(index=False))


### 场景 2：告警传感器 Top10

In [ ]:
df2 = conn.execute('''
    SELECT mine, tag, high_value_count, avg_value
    FROM dwa_alarm ORDER BY high_value_count DESC LIMIT 10
''').df()
print(f'场景2 - Top10 告警量：{df2["high_value_count"].sum()} 次')
print(df2.to_string(index=False))


### 场景 3：月度煤质

In [ ]:
df3 = conn.execute('''
    SELECT MINE_CODE, month, SAMPLE_TYPE,
           avg_ash_content, avg_gross_calorific
    FROM dwa_quality ORDER BY month DESC LIMIT 10
''').df()
print(f'场景3 - 月度煤质：{len(df3)} 条记录')
print(df3.to_string(index=False))


### 场景 4：产销对比（⚠️ 需业务自己写 JOIN）

In [ ]:
print('场景4 - 产销对比：⚠️ 当前 3 张 DWA 为单系统汇总，')
print('跨系统产销对比需 JOIN PI 生产数据，Phase 2 交付后自动解决。')
print('dwa_sales_daily（SAP）无法与 dwa_tag_alarm（PI）直接关联。')


## 故障排查

| 现象 | 原因 | 处理 |
|------|------|------|
| DuckDB 查出来为空 | DWA 表未生成 | 先跑 `build_dwa_models.py --layer dwa` |
| Delta 路径读不到 | 目录不存在 | `ls data/lakehouse/dwa/` 确认 |
| 告警排名全是 0 | 数据无 >8000 的值 | 查全量数据确认 |
| 月度煤质字段为空 | 列名大小写问题 | `df.columns.tolist()` 确认 |

**自查清单**：表不存在 → 列不存在 → 语法错误（路径需单引号）→ 内存不足


## 快速命令

```bash
# 构建 DWA 表
uv run python scripts/build_dwa_models.py --layer dwa

# DuckDB CLI 即席查询
duckdb -c "SELECT * FROM 'data/lakehouse/dwa/sap_erp/dwa_sales_daily' LIMIT 5;"
duckdb -c "SELECT * FROM 'data/lakehouse/dwa/pi_system/dwa_tag_alarm' ORDER BY high_value_count DESC LIMIT 5;"
duckdb -c "SELECT * FROM 'data/lakehouse/dwa/lims/dwa_coal_quality' ORDER BY month DESC LIMIT 5;"
```

**SQL 模板**（销售趋势）：
```sql
SELECT sale_date, order_count, total_amount
FROM 'data/lakehouse/dwa/sap_erp/dwa_sales_daily/'
ORDER BY sale_date DESC LIMIT 30;
```


## 诚实声明：当前 DWA 的边界

3 张 DWA 宽表均为**单系统**汇总：

- `dwa_sales_daily` ← SAP VBAK
- `dwa_tag_alarm` ← PI tags（告警，非生产量）
- `dwa_coal_quality` ← LIMS samples

**跨系统产销对比**需要 JOIN PI 生产实绩 × SAP 销售 × LIMS × KNA1，
依赖 **Phase 2 主数据标准化**（统一矿井/客户编码维表）。当前需业务自己写 SQL JOIN。


## Module 6 总结

| 场景 | 数据源 | 可达性 |
|------|--------|-------|
| 销售趋势 | `dwa_sales_daily` | ✅ |
| 告警传感器排名 | `dwa_tag_alarm` | ✅ |
| 月度煤质 | `dwa_coal_quality` | ✅ |
| 产销对比 | 多系统 JOIN | ⚠️ Phase 2 |

**前置模块**：[模块五 DWA 宽表](module5.ipynb)
